## Creating the final feature set for the social dimension

This notebook has two sections: one before exploratory data analysis (EDA) and a small section after EDA. The EDA itself was done in another notebook: `social_dimension_EDA.ipynb`in the notebooks/EDA directory.

In this notebook, the features we want to use for modeling are extracted from the `social_panel_clean.csv` that we created with the `social_data_cleaning.ipynb`.

### Before EDA

First we create the features that we explore later.

In [1]:
# import libraries

import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# defining input and output (= path with the file we want to create as the goal of this notebook)

INPUT_FILE = Path("../../data/social/raw/social_panel_clean.csv")
OUTPUT_FILE = Path("../../data/social/final_variables/final_social_features.csv")

In [3]:
# let's read our previously created csv file with the aggregated social dimension data

df = pd.read_csv(
    INPUT_FILE,
    dtype={"plr_key": "string"}     # we have to read this as string, so the "0" at the beginning doesn't get lost
)

df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year

df = df.sort_values(["plr_key", "year"]).reset_index(drop=True)

df.shape

C:\Users\valer\AppData\Local\Temp\ipykernel_18220\593467558.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["year"] = df["date"].dt.year


(3252, 120)

In [4]:
df = df.copy()

In [5]:
# just a quick check, if we have the right length, no duplicates and if the plr_key has a length of 8 digits

print(df["year"].value_counts().sort_index())
print(df.duplicated(subset=["plr_key", "year"]).sum())
print(df["plr_key"].str.len().value_counts())

year
2020    542
2021    542
2022    542
2023    542
2024    542
2025    542
Name: count, dtype: int64
0
plr_key
8    3252
Name: count, dtype: Int64


In [ ]:
# ok, let's calculate our base features now

# we need the averaged resident count (=population_exposure) for calculation of fluctuation:
df["res_count_previous_year"] = df.groupby("plr_key")["res_count"].shift(1)

df["population_exposure"] = (
    df["res_count_previous_year"] + df["res_count"]
) / 2


# share of young to middle aged residents:
df["young_to_middle_adult_share"] = (
    df["age_4"] + df["age_5"]
) / df["res_count"]


# household sizes / share of households without children and single parents:
df["single_person_hh_share"] = (
    df["single_pers_hh_count"] / df["households_total"]
)

df["average_household_size"] = df["avg_hh_size"]

df["households_without_minor_children_share"] = (
    1 - (df["hh_with_child_total_count"] / df["households_total"])
)

df["single_parent_household_share"] = (
    df["single_parent_hh_count"] / df["households_total"]
)


# fluctuation of the population:
df["internal_migration_volume_rate"] = (
    df["int_in_migr_total_count"] + df["int_out_migr_total_count"]
) / df["population_exposure"]

df["internal_net_migration_rate"] = (
    df["int_in_migr_total_count"] - df["int_out_migr_total_count"]
) / df["population_exposure"]


# share of unemployed people and people receiving transfer benefits:
df["unemployment_share"] = (
    df["s1_unempl_status_indicator"] / 100
)

df["transfer_benefit_share"] = (
    df["s3_transfer_benefit_status_indicator"] / 100
)


# median income, just to be able to compare with clusters later on:
df["median_income"] = df["median_income_eur"]

In [ ]:
df.head()

In [ ]:
# now we need a helper funciton to get the year values

def get_year_values(data, year, columns):
    year_df = data.loc[
        data["year"].eq(year),
        ["plr_key", "plr_name", *columns]
    ].copy()

    year_df = year_df.rename(
        columns={col: f"{col}_{year}" for col in columns}
    )

    return year_df

In [ ]:
# now let's create a data frame where we have the plr_key only once and the years as addition to the variable names


final_df = get_year_values(df, 2025, [])    # takes all the plr_keys and plr_names from 2025 and serves as a base table

# what we need from which year:

tables_to_merge = [
    (
        2025,
        [
            "young_to_middle_adult_share",
            "internal_migration_volume_rate",
            "internal_net_migration_rate",
        ],
    ),
    (
        2021,
        [
            "young_to_middle_adult_share",
            "internal_migration_volume_rate",
            "internal_net_migration_rate",
        ],
    ),
    (
        2024,
        [
            "single_person_hh_share",
            "average_household_size",
            "households_without_minor_children_share",
            "unemployment_share",
            "transfer_benefit_share",
            "single_parent_household_share",
        ],
    ),
    (
        2021,
        [
            "single_person_hh_share",
            "average_household_size",
            "households_without_minor_children_share",
            "single_parent_household_share",
        ],
    ),
    (
        2020,
        [
            "unemployment_share",
            "transfer_benefit_share",
        ],
    ),
    (
        2023,
        [
            "median_income",
        ],
    ),
]


for year, columns in tables_to_merge:
    year_df = get_year_values(df, year, columns)    # only gets the rows of the year we need and only the variables we need

    year_df = year_df.drop(columns="plr_name")      # plr_name is already in the base table, so we can drop that

    final_df = final_df.merge(      # attaches the rows to the final_df
        year_df,
        on="plr_key",
        how="left"
    )

final_df.shape

In [ ]:
# now we can calculate the change variables (absolute differences between the two years)

final_df["young_to_middle_adult_share_change_2021_2025"] = (
    final_df["young_to_middle_adult_share_2025"]
    - final_df["young_to_middle_adult_share_2021"]
)

final_df["single_person_hh_share_change_2021_2024"] = (
    final_df["single_person_hh_share_2024"]
    - final_df["single_person_hh_share_2021"]
)

final_df["average_household_size_change_2021_2024"] = (
    final_df["average_household_size_2024"]
    - final_df["average_household_size_2021"]
)

final_df["households_without_minor_children_share_change_2021_2024"] = (
    final_df["households_without_minor_children_share_2024"]
    - final_df["households_without_minor_children_share_2021"]
)

final_df["internal_migration_volume_rate_change_2021_2025"] = (
    final_df["internal_migration_volume_rate_2025"]
    - final_df["internal_migration_volume_rate_2021"]
)

final_df["internal_net_migration_rate_change_2021_2025"] = (
    final_df["internal_net_migration_rate_2025"]
    - final_df["internal_net_migration_rate_2021"]
)

final_df["unemployment_share_change_2020_2024"] = (
    final_df["unemployment_share_2024"]
    - final_df["unemployment_share_2020"]
)

final_df["transfer_benefit_share_change_2020_2024"] = (
    final_df["transfer_benefit_share_2024"]
    - final_df["transfer_benefit_share_2020"]
)

final_df["single_parent_household_share_change_2021_2024"] = (
    final_df["single_parent_household_share_2024"]
    - final_df["single_parent_household_share_2021"]
)

In [ ]:
# and now we only keep the rows that we need for the modeling

final_columns = [
    "plr_key",
    "plr_name",
    "median_income_2023",
    "young_to_middle_adult_share_2025",
    "young_to_middle_adult_share_change_2021_2025", 
    "single_person_hh_share_2024",
    "single_person_hh_share_change_2021_2024",
    "average_household_size_2024",
    "average_household_size_change_2021_2024",
    "households_without_minor_children_share_2024",
    "households_without_minor_children_share_change_2021_2024",
    "internal_migration_volume_rate_2025",
    "internal_migration_volume_rate_change_2021_2025",
    "internal_net_migration_rate_2025",
    "internal_net_migration_rate_change_2021_2025",
    "unemployment_share_2024",
    "unemployment_share_change_2020_2024",
    "transfer_benefit_share_2024",
    "transfer_benefit_share_change_2020_2024",
    "single_parent_household_share_2024",
    "single_parent_household_share_change_2021_2024",
]

final_df = final_df[final_columns]

final_df.head()

# looks good already, but let's maybe round the numbers and also rename plr_key

In [ ]:
# let's rename plr_key to plr_id, to have a unified name for it =)

final_df = final_df.rename({"plr_key" : "plr_id"}, axis=1)

final_df.head()

In [ ]:
# and now let's round the numbers to three digits

numeric_cols = final_df.select_dtypes(include="number").columns

final_df[numeric_cols] = final_df[numeric_cols].round(3)

final_df.head()

In [ ]:
# quick quality check (we expect a shape of 542 rows, as we have 542 PLRs, 21 columns, as we have 21 features, a plr_id length of 8, no duplicates and not too many missing values)

print("Shape:", final_df.shape)
print("Duplicate PLR IDs:", final_df.duplicated(subset=["plr_id"]).sum())
print("PLR ID lengths:")
print(final_df["plr_id"].str.len().value_counts())

missing_report = (
    final_df
    .isna()
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

missing_report.columns = ["column_name", "missing_count"]

missing_report

In [ ]:
# and now we create our final output csv file with all the variables we need =)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

final_df.to_csv(
    OUTPUT_FILE,
    index=False
)

OUTPUT_FILE

### After EDA

For EDA, see notebook `social_dimension_EDA.ipynb`.

We found out that there are high correlations (using spearman correlation) between features (threshold: > 0.79), so we decided to drop:
- households_without_minor_children_share_2024 and single_person_hh_share_2024 &rarr; they are correlated with average_household_size_2024 and the average household size includes both of these, so we keep it
- unemployment_share_2024 &rarr; is correlated with transfer_benefit_share_2024 and the latter includes more cases, so we keep transfer benefit share



In [6]:


df_for_modeling = pd.read_csv("../../data/social/final_variables_for_EDA/final_social_features.csv", dtype={"plr_id": "string"})


df_for_modeling.head()



,plr_id,plr_name,median_income_2023,young_to_middle_adult_share_2025,young_to_middle_adult_share_change_2021_2025,single_person_hh_share_2024,single_person_hh_share_change_2021_2024,average_household_size_2024,average_household_size_change_2021_2024,households_without_minor_children_share_2024,...,internal_migration_volume_rate_2025,internal_migration_volume_rate_change_2021_2025,internal_net_migration_rate_2025,internal_net_migration_rate_change_2021_2025,unemployment_share_2024,unemployment_share_change_2020_2024,transfer_benefit_share_2024,transfer_benefit_share_change_2020_2024,single_parent_household_share_2024,single_parent_household_share_change_2021_2024
0,01100101,Stülerstraße,4721.0,0.460,0.000,0.624,-0.004,1.60,0.02,0.888,...,0.236,0.025,-0.056,-0.043,0.031,-0.011,0.080,-0.000,0.027,0.005
1,01100102,Großer Tiergarten,6279.0,0.479,0.003,0.535,-0.010,1.77,0.02,0.870,...,0.199,0.017,0.008,0.006,0.009,-0.002,0.027,-0.008,0.025,-0.011
2,01100103,Lützowstraße,4895.0,0.455,0.022,0.603,0.006,1.70,0.00,0.866,...,0.172,0.005,0.005,-0.002,0.037,-0.016,0.118,-0.037,0.035,-0.001
3,01100104,Körnerstraße,5062.0,0.507,-0.020,0.584,0.006,1.80,-0.01,0.845,...,0.215,0.034,0.003,0.017,0.053,-0.009,0.144,-0.036,0.034,-0.004
4,01100205,Wilhelmstraße,5384.0,0.493,-0.003,0.548,0.043,1.73,-0.06,0.885,...,0.243,-0.038,-0.035,-0.049,0.025,-0.004,0.047,-0.016,0.023,-0.005


In [ ]:
# let's remove the features from the final dataset!

df_for_modeling = df_for_modeling.drop(["households_without_minor_children_share_2024",
    "single_person_hh_share_2024", "unemployment_share_2024"], axis=1)

df_for_modeling.head()

,plr_id,plr_name,median_income_2023,young_to_middle_adult_share_2025,young_to_middle_adult_share_change_2021_2025,single_person_hh_share_change_2021_2024,average_household_size_2024,average_household_size_change_2021_2024,households_without_minor_children_share_change_2021_2024,internal_migration_volume_rate_2025,internal_migration_volume_rate_change_2021_2025,internal_net_migration_rate_2025,internal_net_migration_rate_change_2021_2025,unemployment_share_change_2020_2024,transfer_benefit_share_2024,transfer_benefit_share_change_2020_2024,single_parent_household_share_2024,single_parent_household_share_change_2021_2024
0,01100101,Stülerstraße,4721.0,0.460,0.000,-0.004,1.60,0.02,-0.019,0.236,0.025,-0.056,-0.043,-0.011,0.080,-0.000,0.027,0.005
1,01100102,Großer Tiergarten,6279.0,0.479,0.003,-0.010,1.77,0.02,0.005,0.199,0.017,0.008,0.006,-0.002,0.027,-0.008,0.025,-0.011
2,01100103,Lützowstraße,4895.0,0.455,0.022,0.006,1.70,0.00,-0.003,0.172,0.005,0.005,-0.002,-0.016,0.118,-0.037,0.035,-0.001
3,01100104,Körnerstraße,5062.0,0.507,-0.020,0.006,1.80,-0.01,0.005,0.215,0.034,0.003,0.017,-0.009,0.144,-0.036,0.034,-0.004
4,01100205,Wilhelmstraße,5384.0,0.493,-0.003,0.043,1.73,-0.06,0.013,0.243,-0.038,-0.035,-0.049,-0.004,0.047,-0.016,0.023,-0.005


In [10]:
df_for_modeling.shape

(542, 18)

In [11]:
# and now let's save it

df_for_modeling.to_csv("../../data/final_datasets/social_features.csv", index=False)